In [2]:
import pandas as pd
import datasets as DS
import numpy as np

books = DS.loadDataset("books.yaml", "./data/STORE1/")
orders = DS.loadDataset("orders.parquet", "./data/STORE1/")
users = DS.loadDataset("users.csv", "./data/STORE1/")

### Books

In [3]:
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 749 entries, 0 to 748
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         749 non-null    int64 
 1   title      749 non-null    str   
 2   author     749 non-null    str   
 3   genre      749 non-null    str   
 4   publisher  747 non-null    str   
 5   year       747 non-null    object
dtypes: int64(1), object(1), str(4)
memory usage: 84.3+ KB


In [4]:
books.head()

,id,title,author,genre,publisher,year
0,23099,Lawrence of Arabia,Amb. September Walter,Western,Matthias Media,1989
1,22627,Into the Wild,Golda Satterfield,Textbook,Charles Scribner's Sons,2007
2,22817,Jurassic Park,"Miss Lorina Kertzmann, Darron Ernser",Biography/Autobiography,HarperTrophy,1953
3,22730,Minecraft,Nathan Hintz,Narrative nonfiction,Hamish Hamilton,2001
4,22682,Dance Dance Dance,Natalie Schuster,Biography/Autobiography,G-Unit Books,1981


In [5]:
books.tail()

,id,title,author,genre,publisher,year
744,23072,Born in the U.S.A.,"Ms. Shanice Feeney, Stephnie Feest",Short story,AndrĂ© Deutsch,1985
745,23156,The Woman in Me,Yasmin Lakin CPA,Science fiction,Basic Books,1980
746,23143,Call of Duty: Modern Warfare 3,Florene Bayer,Comic/Graphic Novel,NavPress,1884
747,22828,Precious Bane,Gov. Donny Dibbert,Crime/Detective,No Starch Press,1968
748,22775,PokĂ©mon Silver,Bruno Pagac,Mythopoeia,Emerald Group Publishing,2019


In [6]:
books.drop_duplicates(inplace = True)
books.shape

(749, 6)

In [7]:
year_str = books["year"].astype(str).str.strip()
idx_correct_years = year_str.str.match(r'^\d{4}$', na=False) # regex for finding only 4 digit number
not_years = books[~idx_correct_years]["year"].unique()       # ~ negates (inverts)
books["year"] = books["year"].replace(not_years, np.nan).astype("Int16")

In [8]:
books["publisher"].isna().sum()

np.int64(2)

In [9]:
books[books["publisher"].isna()]

,id,title,author,genre,publisher,year
30,22847,Amadeus,Mara Reynolds,Comic/Graphic Novel,NaN,2015
157,22708,PokĂ©mon Sword,"Tracey Roob, Keshia West V, Ms. Petrina Kohler...",Folklore,NaN,2026


### Users

In [10]:
users.drop_duplicates(inplace = True)
users.info()
users.shape

<class 'pandas.DataFrame'>
RangeIndex: 3394 entries, 0 to 3393
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       3394 non-null   int64
 1   name     3394 non-null   str  
 2   address  3261 non-null   str  
 3   phone    3394 non-null   str  
 4   email    3394 non-null   str  
dtypes: int64(1), str(4)
memory usage: 466.0 KB


(3394, 5)

In [11]:
def showDuplicated(data: pd.DataFrame):
    for col in data.columns:
        print(f"Duplicated in {col}:", data[col].dropna().duplicated().sum())

# print("Duplicated users id:", users["id"].duplicated().sum())
# print("Duplicated users name:", users["name"].duplicated().sum())
# print("Duplicated users address:", users["address"].duplicated().sum())
# print("Duplicated users phone:", users["phone"].duplicated().sum())
# print("Duplicated users email:", users["email"].duplicated().sum())

showDuplicated(users)

Duplicated in id: 0
Duplicated in name: 121
Duplicated in address: 159
Duplicated in phone: 139
Duplicated in email: 129


In [12]:
def showMissing(data: pd.DataFrame):
    for col in data.columns:
        print(f"Missing in {col}:", data[col].isna().sum())

# print("Missing users id:", users["id"].isna().sum())
# print("Missing users name:", users["name"].isna().sum())
# print("Missing users address:", users["address"].isna().sum())
# print("Missing users phone:", users["phone"].isna().sum())
# print("Missing users email:", users["email"].isna().sum())

showMissing(users)

Missing in id: 0
Missing in name: 0
Missing in address: 133
Missing in phone: 0
Missing in email: 0


In [13]:
users["address"] = users["address"].str.strip().replace('', None)
print("Missing users address:", users["address"].isna().sum())

Missing users address: 163


In [14]:
users["phone"] = users["phone"].str.strip().str.replace(r'\D', '', regex = True)
users["phone"].head()

0    9053145984
1    3139701817
2    3632513408
3    3322562176
4    5620314648
Name: phone, dtype: str

### Orders

In [15]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 10493 entries, 0 to 10492
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          10493 non-null  int64
 1   user_id     10493 non-null  int64
 2   book_id     10493 non-null  int64
 3   quantity    10493 non-null  int32
 4   unit_price  10493 non-null  str  
 5   timestamp   10493 non-null  str  
 6   shipping    7913 non-null   str  
dtypes: int32(1), int64(3), str(3)
memory usage: 947.5 KB


In [16]:
showDuplicated(orders)
showMissing(orders)

Duplicated in id: 0
Duplicated in user_id: 7252
Duplicated in book_id: 9744
Duplicated in quantity: 10488
Duplicated in unit_price: 4449
Duplicated in timestamp: 0
Duplicated in shipping: 6178
Missing in id: 0
Missing in user_id: 0
Missing in book_id: 0
Missing in quantity: 0
Missing in unit_price: 0
Missing in timestamp: 0
Missing in shipping: 2580


In [17]:
def isEuro(price:str) -> bool:
    if "€" in price:
        return True
    
    price = price.strip().lower()
    
    if "eur" in price:
        return True
    
    return False

def isDollar(price:str) -> bool:
    if "$" in price:
        return True
    
    price = price.strip().lower()
    
    if "usd" in price:
        return True
    
    return False

len(orders[~orders["unit_price"].apply(lambda p: isEuro(p) or isDollar(p))]) # see if after division no other price formats are left

0

In [18]:
orders["quantity"].value_counts() # all good there

quantity
1    7393
2    1747
3     669
4     353
5     331
Name: count, dtype: int64

In [19]:
orders["timestamp"].head(10)

0           09:28,10-Dec-2025
1    2025-06-25;05:00:01 A.M.
2    10:31:43 P.M. 2026-01-14
3        03:02:34;17-JUL-2026
4            2025-08-17 12:25
5    Fri Apr 10 15:32:26 2026
6     20-Jul-2025;01:00:12 AM
7     15-September-2025,16:55
8      2025-11-28 05:13:06 am
9           2026-01-08, 14:26
Name: timestamp, dtype: str

In [75]:
from dateutil import parser

def parseDate(date:str) -> str | pd.api.typing.NaTType:
    date = date.replace(",", " ")                                               # main culprit - ","
    try:
        return str(parser.parse(date, ignoretz=True, fuzzy=True)).split(".")[0] # throwing out miliseconds
    except (ValueError, TypeError, OverflowError):
        return pd.NaT

orders["date"] = pd.to_datetime(orders["timestamp"].map(parseDate), errors="coerce").dt.strftime("%Y-%m-%d")
orders["date"].isna().sum()

np.int64(0)